In [2]:
!jupyter kernelspec list

Available kernels:
  python3    C:\Users\majun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\share\jupyter\kernels\python3


In [1]:
import sys
from pathlib import Path

# adjust to your imaginaire root if running standalone
_PROJECT_ROOT = Path("/home/theperfectbanana/Yu/comparisons/models/UNIT_pix2pixHD")
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from imaginaire.config import Config
from imaginaire.datasets.h5_mri import H5MRIDataset
from get_h5_dataloaders import _parse_aug  # reuse your existing aug parser

ModuleNotFoundError: No module named 'numpy'

In [ ]:
config_path = "/home/theperfectbanana/Yu/comparisons/Results/munit_run/unit_mri.yaml"
file_list_path = "/home/theperfectbanana/Yu/comparisons/Results/munit_run/val_files.txt"  # or train_files.txt

cfg = Config(config_path)

h5_files = [Path(p) for p in Path(file_list_path).read_text().splitlines() if p.strip()]
aug = _parse_aug(cfg.data.val.augmentations)  # swap to cfg.data.train.augmentations for the train set

dataset = H5MRIDataset(
    h5_files=h5_files,
    modality_a=cfg.data.modality_a,
    modality_b=cfg.data.modality_b,
    augmentations=aug,
)

loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)
print(f"Dataset ready — {len(dataset)} paired slices")

In [ ]:
batch = next(iter(loader))
print("batch keys:", list(batch.keys()))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f"  {k}: shape={tuple(v.shape)} dtype={v.dtype} "
              f"min={v.min().item():.3f} max={v.max().item():.3f}")

In [ ]:
def to_display(img_tensor):
    """
    img_tensor: (C, H, W) tensor, assumed normalized to [-1, 1]
    (matches the .clamp(-1, 1) convention used in evaluate_h5_munit.py).
    Returns an (H, W, C) or (H, W) numpy array in [0, 1] for imshow.
    """
    img = img_tensor.detach().cpu().float()
    img = (img.clamp(-1, 1) + 1) / 2  # [-1,1] -> [0,1]
    img = img.numpy()
    if img.shape[0] == 1:
        return img[0]              # grayscale
    return np.transpose(img, (1, 2, 0))  # CHW -> HWC


def show_paired_batch(batch, key_a="images_a", key_b="images_b", n=4, cmap="gray"):
    images_a = batch[key_a]
    images_b = batch[key_b]
    n = min(n, images_a.shape[0])

    fig, axes = plt.subplots(n, 2, figsize=(6, 3 * n))
    if n == 1:
        axes = axes[None, :]

    for i in range(n):
        axes[i, 0].imshow(to_display(images_a[i]), cmap=cmap)
        axes[i, 0].set_title(f"modality A (idx {i})")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(to_display(images_b[i]), cmap=cmap)
        axes[i, 1].set_title(f"modality B (idx {i})")
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()


show_paired_batch(batch, n=4)